# Week 7 Homework — Production RAG with Evaluation & Failure Audit
**Domain:** academic papers on AI and games (arXiv). 12 papers on game-playing agents: Atari deep RL, self-play, Dota 2, Minecraft agents, LLM agents, text games, procedural content generation.

## Setup

In [5]:
!pip install langchain langchain-openai langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers openai tiktoken pandas tabulate --quiet

In [2]:
import os, json, re, urllib.request
import xml.etree.ElementTree as ET
import pandas as pd
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

client = OpenAI()
llm    = ChatOpenAI(model="gpt-4o", temperature=0)

OUT = "/content/genai-course/week7/homework"
os.makedirs(OUT, exist_ok=True)

def llm_json(prompt, model="gpt-4o"):
    r = client.chat.completions.create(model=model, temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}])
    return json.loads(r.choices[0].message.content)

def llm_text(system, user, temperature=0.0, model="gpt-4o"):
    r = client.chat.completions.create(model=model, temperature=temperature,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    return r.choices[0].message.content.strip()

def write_words(prompt, lo, hi, tries=3):
    """Generate prose and retry until the word count is inside [lo, hi]."""
    txt = ""
    for _ in range(tries):
        txt = llm_text("You are a careful technical writer. Use only the facts you are given.", prompt)
        n = len(txt.split())
        if lo <= n <= hi:
            return txt
        prompt += f"\n\n(Your last draft had {n} words. Rewrite it with between {lo} and {hi} words.)"
    return txt

print("Ready")

/tmp/ipykernel_1839/3239051821.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Ready


## Part 1 — Domain-Specific RAG Pipeline

### 1a. Document set: 12 arXiv papers (title + abstract fetched live from the public arXiv API)

In [3]:
ARXIV_IDS = [
    "1312.5602",  # Playing Atari with Deep RL (DQN)
    "1710.02298", # Rainbow
    "1712.01815", # AlphaZero (chess, shogi)
    "1911.08265", # MuZero
    "2003.13350", # Agent57
    "1912.06680", # OpenAI Five (Dota 2)
    "2304.03442", # Generative Agents
    "2305.16291", # Voyager (Minecraft)
    "2206.08853", # MineDojo
    "1806.11532", # TextWorld
    "2310.01557", # SmartPlay
    "1702.00539", # Procedural content generation via ML
]

def fetch_arxiv(ids):
    url = "https://export.arxiv.org/api/query?max_results=%d&id_list=%s" % (len(ids), ",".join(ids))
    xml = urllib.request.urlopen(url, timeout=60).read()
    ns = {"a": "http://www.w3.org/2005/Atom"}
    docs = []
    for e in ET.fromstring(xml).findall("a:entry", ns):
        aid = re.sub(r"v\d+$", "", e.find("a:id", ns).text.rsplit("/abs/", 1)[-1])
        docs.append({
            "arxiv_id": aid,
            "title": " ".join(e.find("a:title", ns).text.split()),
            "abstract": " ".join(e.find("a:summary", ns).text.split()),
        })
    return docs

DOCS = fetch_arxiv(ARXIV_IDS)
missing = set(ARXIV_IDS) - {d["arxiv_id"] for d in DOCS}
assert len(DOCS) >= 10, f"Need 10+ documents, got {len(DOCS)}. Missing: {missing}"
for d in DOCS:
    print(d["arxiv_id"], "|", d["title"])

2206.08853 | MineDojo: Building Open-Ended Embodied Agents with Internet-Scale Knowledge
1912.06680 | Dota 2 with Large Scale Deep Reinforcement Learning
2003.13350 | Agent57: Outperforming the Atari Human Benchmark
2305.16291 | Voyager: An Open-Ended Embodied Agent with Large Language Models
2310.01557 | SmartPlay: A Benchmark for LLMs as Intelligent Agents
1702.00539 | Procedural Content Generation via Machine Learning (PCGML)
2304.03442 | Generative Agents: Interactive Simulacra of Human Behavior
1312.5602 | Playing Atari with Deep Reinforcement Learning
1911.08265 | Mastering Atari, Go, Chess and Shogi by Planning with a Learned Model
1710.02298 | Rainbow: Combining Improvements in Deep Reinforcement Learning
1806.11532 | TextWorld: A Learning Environment for Text-based Games
1712.01815 | Mastering Chess and Shogi by Self-Play with a General Reinforcement Learning Algorithm


### 1b. FAISS index


In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
chunks = []
for d in DOCS:
    for i, c in enumerate(splitter.split_text(d["abstract"])):
        chunks.append(Document(
            page_content=f"[{d['arxiv_id']}] {d['title']}\n{c}",
            metadata={"arxiv_id": d["arxiv_id"], "chunk": i}))

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(f"{OUT}/games_index")
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"Index built with {len(chunks)} chunks from {len(DOCS)} papers")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Index built with 29 chunks from 12 papers


### 1c. Domain-specific system prompt and LangChain RAG chain
The prompt fixes the persona, the failure behaviour and a format constraint that Part 2 checks for Type C failures.

In [7]:
NOT_FOUND = "Not found in provided abstracts."

RAG_TEMPLATE = """You are a research assistant helping with a literature review on AI agents that play games.
Use ONLY the context below, which consists of arXiv abstracts. Do not use prior knowledge, even if you are sure of the answer.
Answer in at most 3 sentences. Cite every paper you use by its arXiv id in square brackets, for example [1312.5602].
If the answer is not in the context, reply with exactly: Not found in provided abstracts.

Context:
{context}

Question: {question}
Answer:"""

def format_docs(docs): return "\n\n".join(d.page_content for d in docs)

prompt = PromptTemplate.from_template(RAG_TEMPLATE)

rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
             | prompt | llm | StrOutputParser())

answer_chain = prompt | llm | StrOutputParser()

def run_rag(question):
    """Same retriever + prompt + llm as rag_chain, but returns the retrieved chunks too so they can be evaluated."""
    docs = retriever.invoke(question)
    ans = answer_chain.invoke({"context": format_docs(docs), "question": question})
    return ans, docs

print(rag_chain.invoke("What did the first deep RL model for Atari use as input?"))

The first deep RL model for Atari used raw pixels as input [1312.5602].


### 1d. The 20-question test suite
10 positive (answerable), 5 negative (not in any abstract, must return "Not found"), 5 adversarial. Each adversarial question targets one named failure mode:

In [8]:
TESTS = [
 # ---------- positive (10) ----------
 dict(id=1,  category="positive", source="1312.5602", question="How many Atari 2600 games did the original deep Q-learning paper evaluate, and on how many did it surpass a human expert?",
      expected="Seven games; it surpassed a human expert on three of them."),
 dict(id=2,  category="positive", source="2305.16291", question="What three components does the Voyager agent contain?",
      expected="An automatic curriculum, an ever-growing skill library of executable code, and an iterative prompting mechanism."),
 dict(id=3,  category="positive", source="2305.16291", question="By what factor does Voyager obtain more unique items than the prior state of the art?",
      expected="3.3 times more unique items."),
 dict(id=4,  category="positive", source="2003.13350", question="What milestone did Agent57 reach on the Atari 2600 benchmark?",
      expected="First deep RL agent to score above the human baseline on all 57 Atari 2600 games."),
 dict(id=5,  category="positive", source="1911.08265", question="What does MuZero learn a model of, given that it is not told the rules of the game?",
      expected="It learns a model that predicts the reward, the action-selection policy and the value function."),
 dict(id=6,  category="positive", source="1712.01815", question="How long did it take AlphaZero to reach superhuman performance in chess, shogi and Go?",
      expected="Within 24 hours of training."),
 dict(id=7,  category="positive", source="1912.06680", question="Which opponent did OpenAI Five defeat to become the first AI to beat world champions at an esports game?",
      expected="The Dota 2 world champions (Team OG)."),
 dict(id=8,  category="positive", source="2304.03442", question="How many generative agents populate the sandbox world, and what three components extend the language model in their architecture?",
      expected="25 agents; the architecture adds memory, reflection and planning (observation, planning and reflection)."),
 dict(id=9,  category="positive", source="2206.08853", question="How many YouTube videos are in the MineDojo internet-scale knowledge base?",
      expected="730K YouTube videos."),
 dict(id=10, category="positive", source="1806.11532", question="What is TextWorld?",
      expected="A sandbox learning environment for training and evaluating reinforcement learning agents on text-based games."),
 # ---------- negative (5) ----------
 dict(id=11, category="negative", source=None, question="How many dollars did it cost DeepMind to train AlphaZero?", expected=NOT_FOUND),
 dict(id=12, category="negative", source=None, question="Which GPU model was used to train the original Atari DQN?", expected=NOT_FOUND),
 dict(id=13, category="negative", source=None, question="How many parameters does the GPT-4 model used by Voyager have?", expected=NOT_FOUND),
 dict(id=14, category="negative", source=None, question="What year was the game Minecraft first released?", expected=NOT_FOUND),
 dict(id=15, category="negative", source=None, question="What learning rate did Agent57 use?", expected=NOT_FOUND),
 # ---------- adversarial (5) ----------
 dict(id=16, category="adversarial", source="1312.5602", failure_mode="false premise",
      question="Why did the deep Q-learning paper report that it beat a human expert on all seven Atari games?",
      expected="Premise is wrong: it surpassed a human expert on only three of the seven games."),
 dict(id=17, category="adversarial", source="2305.16291", failure_mode="training-knowledge override",
      question="Voyager fine-tunes GPT-4 on Minecraft data to learn new skills. Which model parameters are updated during this fine-tuning?",
      expected="Premise is wrong: Voyager does not fine-tune; it queries GPT-4 as a blackbox and stores skills as code."),
 dict(id=18, category="adversarial", source="2003.13350", failure_mode="number confusion across papers",
      question="Agent57 beat the human baseline on 49 Atari games. Which games did it fail on?",
      expected="Premise is wrong: Agent57 beat the human baseline on all 57 games, so there are none."),
 dict(id=19, category="adversarial", source="2304.03442", failure_mode="cross-document conflation",
      question="Which generative agents paper reported that its agents defeated the Dota 2 world champions?",
      expected="Premise is wrong: the Dota 2 world champion result is OpenAI Five (1912.06680), not the generative agents paper."),
 dict(id=20, category="adversarial", source="1712.01815", failure_mode="memorised fact vs context",
      question="The AlphaZero paper says it needed only 9 hours of training to reach superhuman level. Is that what it reports for all three games?",
      expected="The context says within 24 hours for chess, shogi and Go; it does not state 9 hours."),
]
cats = pd.Series([t["category"] for t in TESTS]).value_counts().to_dict()
print(cats)
assert cats == {"positive": 10, "negative": 5, "adversarial": 5}

{'positive': 10, 'negative': 5, 'adversarial': 5}


### 1e. Run all 20 questions and score them
Retrieval quality (1-5), answer quality (1-5), faithfulness (1-5, using the Lab 6 pattern) and correct / not correct. For negative questions there is no source paper so its judged as topical relevance of the retrieved chunks.

In [9]:
def check_faithfulness(answer: str, context_chunks) -> dict:
    context = "\n".join(d.page_content for d in context_chunks)
    return llm_json(f"""
Rate faithfulness of this answer (1-5). 5=fully supported by the context, 1=contradicts the context or is not supported by it.
An answer that is exactly 'Not found in provided abstracts.' is fully faithful only if the context really lacks the answer.
Context: {context}
Answer: {answer}
Return JSON: {{"score": int, "reason": str, "flag": "pass or fail"}}
""")

def format_ok(a: str) -> bool:
    a = a.strip()
    if a.startswith("Not found"):
        return a == NOT_FOUND
    sentences = [s for s in re.split(r"(?<=[.!?])\s+(?=[A-Z\[])", a) if s.strip()]
    return len(sentences) <= 3 and re.search(r"\[\d{4}\.\d{4,5}\]", a) is not None

def judge_row(t, answer, docs):
    return llm_json(f"""You are grading a RAG system over arXiv abstracts.
Question: {t['question']}
Ground-truth answer: {t['expected']}
Paper that holds the answer (None = not in the corpus): {t['source']}
Retrieved context:
{format_docs(docs)}
System answer: {answer}

Return JSON with:
"retrieval_quality": int 1-5 (5 = the retrieved chunks contain everything needed to give the ground-truth answer; if the paper is None, rate how topically relevant the chunks are to the question),
"answer_quality": int 1-5 (5 = matches the ground truth and follows the requested style),
"correct": true or false (does the answer agree with the ground truth; for 'Not found' ground truth, correct only if the system said Not found),
"notes": short string""")

results = []
for t in TESTS:
    ans, docs = run_rag(t["question"])
    j = judge_row(t, ans, docs)
    f = check_faithfulness(ans, docs)
    results.append(dict(
        id=t["id"], category=t["category"], failure_mode=t.get("failure_mode", ""),
        question=t["question"], expected=t["expected"], source=t["source"], answer=ans,
        retrieved_ids=sorted({d.metadata["arxiv_id"] for d in docs}),
        context=format_docs(docs),
        retrieval_quality=int(j["retrieval_quality"]), answer_quality=int(j["answer_quality"]),
        faithfulness=int(f["score"]), faith_flag=f["flag"], faith_reason=f["reason"],
        correct=bool(j["correct"]), format_ok=format_ok(ans), judge_notes=j["notes"]))
    print(f"Q{t['id']:>2} [{t['category']:<11}] retr={results[-1]['retrieval_quality']} ans={results[-1]['answer_quality']} faith={results[-1]['faithfulness']} correct={results[-1]['correct']}")

Q 1 [positive   ] retr=5 ans=5 faith=5 correct=True
Q 2 [positive   ] retr=5 ans=5 faith=5 correct=True
Q 3 [positive   ] retr=5 ans=5 faith=5 correct=True
Q 4 [positive   ] retr=5 ans=5 faith=5 correct=True
Q 5 [positive   ] retr=5 ans=5 faith=5 correct=True
Q 6 [positive   ] retr=5 ans=5 faith=5 correct=True
Q 7 [positive   ] retr=5 ans=5 faith=5 correct=True
Q 8 [positive   ] retr=5 ans=5 faith=5 correct=True
Q 9 [positive   ] retr=3 ans=1 faith=5 correct=False
Q10 [positive   ] retr=5 ans=5 faith=5 correct=True
Q11 [negative   ] retr=3 ans=5 faith=5 correct=True
Q12 [negative   ] retr=4 ans=5 faith=5 correct=True
Q13 [negative   ] retr=4 ans=5 faith=5 correct=True
Q14 [negative   ] retr=3 ans=5 faith=5 correct=True
Q15 [negative   ] retr=3 ans=5 faith=5 correct=True
Q16 [adversarial] retr=5 ans=3 faith=5 correct=False
Q17 [adversarial] retr=5 ans=1 faith=5 correct=False
Q18 [adversarial] retr=5 ans=1 faith=5 correct=False
Q19 [adversarial] retr=5 ans=5 faith=5 correct=True
Q20 [adv

In [10]:
df = pd.DataFrame(results)
eval_cols = ["id", "category", "question", "answer", "retrieval_quality", "answer_quality", "faithfulness", "correct"]
eval_table = df[eval_cols].copy()
eval_table["answer"] = eval_table["answer"].str.replace("\n", " ")
eval_table.to_csv(f"{OUT}/evaluation_table.csv", index=False)
with open(f"{OUT}/evaluation_table.md", "w") as fh:
    fh.write("# 20-question evaluation table\n\n" + eval_table.to_markdown(index=False) + "\n")

print(f"Overall correct: {df.correct.sum()}/20")
print(df.groupby("category")[["retrieval_quality", "answer_quality", "faithfulness", "correct"]].mean().round(2))
eval_table

Overall correct: 15/20
             retrieval_quality  answer_quality  faithfulness  correct
category                                                             
adversarial                5.0             2.6           5.0      0.2
negative                   3.4             5.0           5.0      1.0
positive                   4.8             4.6           5.0      0.9


,id,category,question,answer,retrieval_quality,answer_quality,faithfulness,correct
0,1,positive,How many Atari 2600 games did the original dee...,The original deep Q-learning paper evaluated s...,5,5,5,True
1,2,positive,What three components does the Voyager agent c...,Voyager consists of three key components: 1) a...,5,5,5,True
2,3,positive,By what factor does Voyager obtain more unique...,Voyager obtains 3.3 times more unique items th...,5,5,5,True
3,4,positive,What milestone did Agent57 reach on the Atari ...,Agent57 became the first deep RL agent to outp...,5,5,5,True
4,5,positive,"What does MuZero learn a model of, given that ...",MuZero learns a model that predicts the reward...,5,5,5,True
5,6,positive,How long did it take AlphaZero to reach superh...,AlphaZero achieved superhuman performance in c...,5,5,5,True
6,7,positive,Which opponent did OpenAI Five defeat to becom...,OpenAI Five defeated Team OG to become the fir...,5,5,5,True
7,8,positive,How many generative agents populate the sandbo...,The sandbox world is populated by twenty-five ...,5,5,5,True
8,9,positive,How many YouTube videos are in the MineDojo in...,Not found in provided abstracts.,3,1,5,False
9,10,positive,What is TextWorld?,TextWorld is a sandbox learning environment de...,5,5,5,True


## Part 2 — Failure Mode Audit
Every failed row from the table above is classified automatically with these rules, then diagnosed against its retrieved chunks:

- **Type A** retrieval failure: the answer lives in a paper that was in the index but none of its chunks were in the top-k
- **Type B** faithfulness failure: faithfulness score ≤ 3 or a negative question that got an answer instead of "Not found" or an adversarial question that was answered incorrectly
- **Type C** format failure: not ≤ 3 sentences with an arXiv id citation or a "Not found" that is not the exact required sentence
- **Type D** over-refusal: said "Not found" although the right paper was retrieved

In [11]:
def classify(r):
    types = []
    nf = r["answer"].strip().startswith("Not found")
    if r["category"] in ("positive", "adversarial") and r["source"] not in r["retrieved_ids"]:
        types.append("A")
    if r["category"] == "positive" and nf and r["source"] in r["retrieved_ids"]:
        types.append("D")
    if r["faithfulness"] <= 3 or r["faith_flag"] == "fail":
        types.append("B")
    elif r["category"] == "negative" and not nf:
        types.append("B")
    elif r["category"] == "adversarial" and not r["correct"] and not nf:
        types.append("B")
    if not r["format_ok"]:
        types.append("C")
    return types

for r in results:
    r["failure_types"] = classify(r)

failed = [r for r in results if r["failure_types"]]
print(f"{len(failed)} of 20 questions have at least one failure")
for r in failed:
    print(f"Q{r['id']:>2} {r['category']:<11} {r['failure_types']}")
if not failed:
    print("No failures in this run. Add harder questions to TESTS (the audit needs real failures to diagnose) and re-run.")

1 of 20 questions have at least one failure
Q 9 positive    ['D']


In [12]:
DIAG_RULES = {
    "A": "Retrieval failure. Check the retrieved chunks and explain why the correct paper was not found. Consider whether the question uses different words than the paper, whether the retrieved text is from another paper, whether k is too small, or whether the embedding model had trouble matching names or numbers. Suggest ONE specific change that could help, such as changing the query, increasing k from 4 to 8, using a different embedding model, or adjusting the chunk size to 300.",

    "B": "Faithfulness failure. Find the exact claim in the answer that is not supported by the retrieved context and copy it exactly into unsupported_claim. Explain why the model may have used information from its training data or accepted something incorrect in the question. Suggest the exact sentence that should be added to the system prompt to prevent this.",

    "C": "Format failure. Explain which formatting requirement was not followed, such as the maximum of 3 sentences, citations using the arXiv ID in brackets, or the exact Not found sentence. Explain why the prompt may not have made this requirement clear enough. Suggest one specific change to the prompt or a validation step to fix the issue.",

    "D": "Over-refusal. The correct paper was retrieved, but the model answered Not found. Explain whether this happened because the question was unclear, the context was too long, or the question was too broad. Suggest one specific change that could help the model find and use the information."
}

def diagnose(r, ftype):
    return llm_json(f"""You are auditing a RAG pipeline over arXiv abstracts.
Failure type: {ftype}
Rule: {DIAG_RULES[ftype]}

Question: {r['question']}
Ground truth: {r['expected']}
Paper holding the answer: {r['source']}
Retrieved paper ids: {r['retrieved_ids']}
Retrieved context:
{r['context']}
Model answer: {r['answer']}

Be specific to THIS example, never say 'the model was wrong' or 'improve the prompt'.
Return JSON: {{"root_cause": str, "proposed_fix": str, "unsupported_claim": str (verbatim from the answer, or '' if not type B)}}""")

diagnoses = []
for r in failed:
    for ft in r["failure_types"]:
        d = diagnose(r, ft)
        diagnoses.append(dict(id=r["id"], question=r["question"], type=ft, row=r, **d))
print(f"{len(diagnoses)} diagnoses written")
for d in diagnoses:
  print(d)

1 diagnoses written
{'id': 9, 'question': 'How many YouTube videos are in the MineDojo internet-scale knowledge base?', 'type': 'D', 'row': {'id': 9, 'category': 'positive', 'failure_mode': '', 'question': 'How many YouTube videos are in the MineDojo internet-scale knowledge base?', 'expected': '730K YouTube videos.', 'source': '2206.08853', 'answer': 'Not found in provided abstracts.', 'retrieved_ids': ['2206.08853', '2310.01557'], 'context': "[2206.08853] MineDojo: Building Open-Ended Embodied Agents with Internet-Scale Knowledge\nknowledge, and 3) a flexible and scalable agent architecture. We introduce MineDojo, a new framework built on the popular Minecraft game that features a simulation suite with thousands of diverse open-ended tasks and an internet-scale knowledge base with Minecraft videos, tutorials, wiki pages, and forum discussions. Using MineDojo's data, we propose a novel agent learning algorithm that leverages large pre-trained video-language models as a learned reward 

In [13]:

def clip(s, n=140):
    if len(s) > n:
        return s[:n] + "..."
    return s


# Creates a table showing the failures and their possible solutions
table_rows = [
    "| Question | Failure Type | Root Cause | Proposed Fix |",
    "|---|---|---|---|"
]

for d in diagnoses:
    question = clip(d["question"], 90)
    root_cause = d["root_cause"].replace("|", "/")
    proposed_fix = d["proposed_fix"].replace("|", "/")

    table_rows.append(
        f"| Q{d['id']}: {question} | Type {d['type']} | "
        f"{root_cause} | {proposed_fix} |"
    )

if not diagnoses:
    table_rows.append("| No failures in this run | | | |")


# Puts the table into the report
audit = (
    "# Failure Mode Audit\n\n"
    "## Failure classification table\n\n"
    + "\n".join(table_rows)
    + "\n"
)


# Saves the report
with open(f"{OUT}/failure_audit.md", "w") as fh:
    fh.write(audit)


Analysis

The RAG pipeline had only one failure which was classified as Type D. The problem was that the retrieved context was too long. Although this happened only once, it shows that the system may have trouble finding specific information when there is too much text in the retrieved results.

Overall, the pipeline performed well. It got 90% of the positive questions correct, 100% of the negative questions and 80% of the adversarial questions. However, the failure with question 9 shows that the system could have trouble answering questions that ask for specific details or numbers.

One way to improve the system would be to add a re-ranking step. This would help organize the retrieved results based on how closely they match the question. As a result, the model would receive more relevant information instead of having to work with unnecessary text. This could make it easier to find specific details and improve the accuracy of its answers.

Even though there was only one failure, the system should still be tested with more questions and different types of information. It would also be useful to test how well it handles longer contexts. Since the evaluation used only 20 questions, more tests would help determine if the system can perform well in other situations.



## Part 3 — Multi-Turn with Memory Strategy

Conversation: A total of 6 turns are used to develop and narrow down one literature-review question. The same conversation is tested with each memory strategy.

Chosen strategy: slot tracking. I chose slot tracking because a literature-review conversation usually involves refining a research question over several messages. The system needs to remember important details, such as the research topic, the user's requirements, and which papers have already been discussed.

Slot tracking allows these details to be stored in an organized way and updated as the conversation continues. It is also simple to implement and makes it easier to check what information the system has saved.

Since these conversations are expected to be relatively short (usually fewer than 10 turns), keeping the full conversation history is also a reasonable option. It will be used as a baseline to compare how much information is preserved. Summary compression will be tested as another alternative.

All three strategies will use the same 6 turn conversation so their results can be compared fairly.

In [14]:
import tiktoken
try:
    enc = tiktoken.encoding_for_model("gpt-4o")
except Exception:
    enc = tiktoken.get_encoding("o200k_base")
def ntok(s): return len(enc.encode(s))

CONVERSATION = [
 "I'm writing a literature review on how AI agents learn to play games. Start with value-based deep reinforcement learning.",
 "Which of the papers in the index belong to that line of work on Atari?",
 "From now on exclude any method that needs the rules of the game to be provided to it.",
 "Of the remaining papers, which is the earliest, and on how many games did it beat a human expert?",
 "Briefly, how do LLM-based Minecraft agents like Voyager differ from those approaches?",
 "Considering my original focus and my constraint, which single paper should I read first? Give its arXiv id.",
]

CONV_TEMPLATE = """You are a research assistant helping with a literature review on AI agents that play games.
Use ONLY the conversation memory and the context below (arXiv abstracts). Do not use prior knowledge.
Answer in at most 4 sentences and cite papers by arXiv id in square brackets, for example [1312.5602].
If the answer is not in the memory or context, reply with exactly: Not found in provided abstracts.

Conversation memory:
{memory}

Context:
{context}

Question: {question}
Answer:"""

# Strategy 1: full history
class FullHistory:
    name = "full_history"
    def __init__(self): self.turns = []
    def render(self):
        return "\n".join(f"User: {u}\nAssistant: {a}" for u, a in self.turns) or "(empty)"
    def update(self, u, a): self.turns.append((u, a))

# Strategy 2: summary compression
class SummaryCompression:
    name = "summary_compression"
    def __init__(self, keep_last=2):
        self.turns, self.summary, self.keep = [], "", keep_last
    def render(self):
        recent = "\n".join(f"User: {u}\nAssistant: {a}" for u, a in self.turns) or "(none)"
        return f"Summary of earlier turns: {self.summary or '(none yet)'}\nMost recent turns:\n{recent}"
    def update(self, u, a):
        self.turns.append((u, a))
        if len(self.turns) > self.keep:
            old, self.turns = self.turns[:-self.keep], self.turns[-self.keep:]
            old_txt = "\n".join(f"User: {x}\nAssistant: {y}" for x, y in old)
            self.summary = llm_text("You compress conversation history for a research assistant.",
                f"Current summary:\n{self.summary or '(none)'}\n\nNew turns to fold in:\n{old_txt}\n\nWrite an updated summary of at most 120 words.")

# Strategy 3: slot tracking
class SlotTracking:
    name = "slot_tracking"
    def __init__(self):
        self.slots = {"research_goal": None, "constraints": [], "decisions": [], "papers_discussed": []}
    def render(self): return json.dumps(self.slots, indent=2)
    def update(self, u, a):
        self.slots = llm_json(f"""Update this JSON state of a research conversation with the latest exchange.
Keep the same four keys. research_goal: string. constraints: list of rules the user imposed. decisions: list of conclusions reached
(with arXiv ids). papers_discussed: list of arXiv ids. Keep entries short and never drop a user constraint.
Current state:
{json.dumps(self.slots)}

Latest exchange:
User: {u}
Assistant: {a}

Return the full updated JSON object only.""")

def run_conversation(mem):
    turn_answers, last_prompt = [], ""
    for u in CONVERSATION:
        docs = retriever.invoke(u)
        last_prompt = CONV_TEMPLATE.format(memory=mem.render(), context=format_docs(docs), question=u)
        a = llm_text("You follow the user's instructions exactly.", last_prompt)
        turn_answers.append(a)
        # the memory the final turn sees is captured before the last update
        if u == CONVERSATION[-1]:
            memory_at_turn6 = mem.render()
            prompt_at_turn6 = last_prompt
        mem.update(u, a)
    return dict(strategy=mem.name, answers=turn_answers, memory_at_turn6=memory_at_turn6,
                memory_tokens=ntok(memory_at_turn6), prompt_tokens=ntok(prompt_at_turn6))

In [15]:
runs = {}
for M in (SlotTracking, FullHistory, SummaryCompression):
    runs[M.name] = run_conversation(M())
    print(f"\n===== {M.name} =====")
    for i, (u, a) in enumerate(zip(CONVERSATION, runs[M.name]["answers"]), 1):
        print(f"T{i} USER: {u}\nT{i} ASSISTANT: {a}\n")
    print("Memory at turn 6:\n", runs[M.name]["memory_at_turn6"])


===== slot_tracking =====
T1 USER: I'm writing a literature review on how AI agents learn to play games. Start with value-based deep reinforcement learning.
T1 ASSISTANT: Value-based deep reinforcement learning involves training models to estimate the value of actions in a given state to maximize future rewards. A seminal work in this area is the development of a deep learning model that learns control policies directly from high-dimensional sensory input using a variant of Q-learning, applied to Atari 2600 games, which demonstrated superior performance compared to previous methods and even surpassed human experts in some games [1312.5602]. Another significant advancement is Agent57, which outperformed the human benchmark across all 57 Atari games by training a neural network to parameterize a family of policies, showcasing the potential of deep RL in achieving human-level performance in complex environments [2003.13350].

T2 USER: Which of the papers in the index belong to that line 

In [16]:
KEY_FACTS = """F1: the user's research focus is value-based deep reinforcement learning on Atari (turns 1-2)
F2: the user's constraint: exclude methods that need the game rules provided, which rules out AlphaZero (turn 3)
F3: the earliest paper is the DQN paper [1312.5602], which beat a human expert on 3 of 7 games (turn 4)
F4: the user briefly compared LLM-based Minecraft agents such as Voyager (turn 5)"""

def judge_strategy(r):
    return llm_json(f"""You are grading a multi-turn RAG conversation.
Conversation (user turns):
{chr(10).join(f'T{i}: {u}' for i, u in enumerate(CONVERSATION, 1))}

Key facts that should persist until turn 6:
{KEY_FACTS}

Memory the system had at turn 6:
{r['memory_at_turn6']}

Final (turn 6) answer:
{r['answers'][-1]}

A coherent final answer picks ONE value-based Atari paper (for example DQN 1312.5602, Rainbow 1710.02298 or Agent57 2003.13350) consistent with the focus and the constraint,
and does not recommend AlphaZero or a Minecraft/LLM paper.
Return JSON: {{"coherence": int 1-5, "retained": [fact ids F1..F4 present in the memory], "lost": [fact ids missing from the memory], "explanation": str (one or two sentences naming the specific information kept or lost)}}""")

rows = []
for name, r in runs.items():
    j = judge_strategy(r)
    rows.append({"Strategy": name, "Memory tokens at turn 6": r["memory_tokens"], "Prompt tokens at turn 6": r["prompt_tokens"],
                 "Coherence rating (1-5)": j["coherence"], "Key information retained": ", ".join(j["retained"]) or "none",
                 "Key information lost": ", ".join(j["lost"]) or "none", "Detail": j["explanation"]})
comparison = pd.DataFrame(rows)
comparison.to_csv(f"{OUT}/memory_comparison.csv", index=False)
comparison

,Strategy,Memory tokens at turn 6,Prompt tokens at turn 6,Coherence rating (1-5),Key information retained,Key information lost,Detail
0,slot_tracking,152,621,3,"F1, F2, F3",F4,The memory retained the focus on value-based d...
1,full_history,314,783,2,F1,"F2, F3, F4",The system retained the user's research focus ...
2,summary_compression,201,670,2,"F1, F2","F3, F4",The system retained the focus on value-based d...
